# Build a Red Team Charter Using LLM-Surfaced Attack Vectors

This notebook supports a 12-minute demo. It loads system context, assembles a structured LLM prompt, reviews sample LLM-generated attack vectors, and points to the completed red team charter.

In [ ]:
from pathlib import Path
from datetime import datetime
import json

DEMO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path("module-3-apply-ai-red-teaming/demo")

api_spec = json.loads((DEMO_ROOT / "data/sentiment_api_spec.json").read_text(encoding="utf-8"))
sample_requests = json.loads((DEMO_ROOT / "data/sample_inference_requests.json").read_text(encoding="utf-8"))
architecture = (DEMO_ROOT / "docs/deployment_architecture.md").read_text(encoding="utf-8")
model_card = (DEMO_ROOT / "docs/model_card.md").read_text(encoding="utf-8")
structured_prompt = (DEMO_ROOT / "prompts/structured_vulnerability_discovery_prompt.md").read_text(encoding="utf-8")

# Structural templates live in docs/ — read-only inputs to the prompt.
attack_vectors_template_path = DEMO_ROOT / "docs/attack_vectors_template.md"
charter_template_path = DEMO_ROOT / "docs/red_team_charter_template.md"
attack_vectors_template = attack_vectors_template_path.read_text(encoding="utf-8")
charter_template = charter_template_path.read_text(encoding="utf-8")

# Generated artifacts go to a separate outputs/ folder with a timestamped name
# so each run preserves prior outputs and never overwrites the source templates.
outputs_dir = DEMO_ROOT / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
attack_vectors_output_path = outputs_dir / f"llm_generated_attack_vectors_{run_timestamp}.md"
charter_output_path = outputs_dir / f"red_team_charter_{run_timestamp}.md"

print(f"Loaded API title: {api_spec['info']['title']}")
print(f"Loaded sample requests: {len(sample_requests)}")
print(f"Outputs will be written to: {outputs_dir}")

## 1. Inspect the System Context

The LLM needs concrete deployment context, not just a model name.

In [ ]:
print(json.dumps(api_spec["paths"], indent=2)[:2000])

In [ ]:
for item in sample_requests:
    text = item["request"]["text"]
    label = item["response"]["label"]
    confidence = item["response"]["confidence"]
    print(f"{label:8} {confidence:.2f} | {text}")

## 2. Assemble the Structured Prompt

The context bundle includes the system context plus templates for both artifacts we want the LLM to generate.

In [ ]:
context_bundle = f"""
{structured_prompt}

## API Specification
```json
{json.dumps(api_spec, indent=2)}
```

## Deployment Architecture
{architecture}

## Model Card
{model_card}

## Sample Inference Records
```json
{json.dumps(sample_requests, indent=2)}
```

## Artifact Template: LLM-Generated Attack Vectors
Use this existing artifact as the required structure and level of detail. Generate fresh content from the provided system context.

{attack_vectors_template}

## Artifact Template: Red Team Charter
Use this existing artifact as the required structure and level of detail. The charter must be based on the generated attack vectors from the first LLM request.

{charter_template}
""".strip()

print(context_bundle[:4000])
print("\n--- prompt truncated for display ---")

## 3. Generate and Save Attack Vectors

This first request generates `docs/llm_generated_attack_vectors.md` from the context bundle.

In [ ]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1"),
)

attack_vector_request = f"""
{context_bundle}

## Generation Task

Generate the first artifact only: `docs/llm_generated_attack_vectors.md`.

Requirements:
- Follow the provided LLM-generated attack vectors template.
- Use only evidence from the API spec, architecture, model card, and sample inference records.
- Include rejected or low-confidence ideas.
- Return Markdown only, with no code fence around the artifact.
""".strip()

attack_vector_response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "system",
            "content": [
                {
                    "type": "input_text",
                    "text": "You are a security analysis assistant helping generate evidence-grounded AI red team planning hypotheses."
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": attack_vector_request,
                }
            ],
        },
    ],
)

generated_attack_vectors = attack_vector_response.output_text.strip() + "\n"
attack_vectors_output_path.write_text(generated_attack_vectors, encoding="utf-8")
print(f"Saved {attack_vectors_output_path}")
print(generated_attack_vectors[:2000])

## 4. Generate and Save the Red Team Charter

This second request appends the generated vectors to the context bundle and asks for `docs/red_team_charter.md` in the provided charter format.

In [ ]:
charter_context_bundle = f"""
{context_bundle}

## Generated Attack Vectors From First LLM Request

{generated_attack_vectors}
""".strip()

charter_request = f"""
{charter_context_bundle}

## Generation Task

Generate the second artifact only: `docs/red_team_charter.md`.

Requirements:
- Follow the provided red team charter template exactly in structure and level of detail.
- Base the Objectives and Success Criteria on the generated attack vectors from the first request.
- Keep the scope, rules of engagement, and deliverables appropriate for a one-business-day pre-production assessment.
- Return Markdown only, with no code fence around the artifact.
""".strip()

charter_response = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "system",
            "content": [
                {
                    "type": "input_text",
                    "text": "You are a security analysis assistant helping generate evidence-grounded AI red team planning artifacts."
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": charter_request,
                }
            ],
        },
    ],
)

generated_charter = charter_response.output_text.strip() + "\n"
charter_output_path.write_text(generated_charter, encoding="utf-8")
print(f"Saved {charter_output_path}")
print(generated_charter[:2000])

## 5. Review Saved Artifacts

After both calls run, review the saved outputs and make any human edits needed before using them as course materials.

In [ ]:
print("# Saved attack vectors\n")
print(attack_vectors_output_path.read_text(encoding="utf-8"))
print("\n# Saved charter\n")
print(charter_output_path.read_text(encoding="utf-8"))